# Lab 3 — Millions of TVs, Who Is Watching?

Name: Anastasiia Movchaniuk | Group: КМ-43

## Setup

In [1]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATA = Path(kagglehub.dataset_download('shkarupylomaxim/adtech-lab1-tv-panel/versions/4'))
programs = pd.read_parquet(DATA / 'programs.parquet')
viewings = pd.read_parquet(DATA / 'viewings.parquet')
daily_weights = pd.read_parquet(DATA / 'daily_weights.parquet')
viewer_attributes = pd.read_parquet(DATA / 'viewer_attributes.parquet')
attributes = pd.read_parquet(DATA / 'attributes.parquet')
acr_sessions = pd.read_parquet(DATA / 'acr_sessions.parquet')
hh_attributes = pd.read_parquet(DATA / 'hh_attributes.parquet')
reference_ratings = pd.read_parquet(DATA / 'reference_ratings.parquet')

100%|██████████| 73.7M/73.7M [00:01<00:00, 69.8MB/s]

Extracting files...


## Task 1.1 — Five questions

**Answer:**

| Dataset | Grain | Key | Coverage | Weight | Provenance |
|---|---|---|---|---|---|
| acr_sessions | One continuous session on one TV | session_id | Opted-in smart TVs, October 14–27, across inputs and networks | Each TV counts once | Synthetic screen fingerprint data modeled on an ACR feed |
| viewings | One viewing session for a panel respondent | viewing_id | October panel viewing on ten networks, with household and person records | Broadcast-day panel weights | Synthetic measurement panel |

The biggest difference is coverage: the smart-TV feed measures an opted-in set of devices, while the weighted panel represents TV households and people.

## Task 1.2 — Size and shape

In [2]:
days = pd.date_range('2024-10-14', '2024-10-27')
period_start = days[0] + pd.Timedelta(hours=6)
period_end = days[-1] + pd.Timedelta(hours=30)
acr = acr_sessions.copy()
for utc, local in [('start_utc', 'start_ts'), ('end_utc', 'end_ts')]:
    acr[local] = pd.to_datetime(acr[utc], utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None)
acr['minutes'] = (acr.end_ts - acr.start_ts).dt.total_seconds() / 60
edited = acr.loc[acr.minutes.ge(1)].copy()

def by_broadcast_day(sessions):
    # Split sessions that cross 06:00 before totaling hours per day.
    remaining = sessions.loc[(sessions.start_ts < period_end) & (sessions.end_ts > period_start)].copy()
    remaining['start_ts'] = remaining.start_ts.clip(lower=period_start)
    remaining['end_ts'] = remaining.end_ts.clip(upper=period_end)
    parts = []
    while not remaining.empty:
        date = (remaining.start_ts - pd.Timedelta(hours=6)).dt.normalize()
        boundary = date + pd.Timedelta(hours=30)
        end = remaining.end_ts.where(remaining.end_ts.le(boundary), boundary)
        hours = (end - remaining.start_ts).dt.total_seconds() / 3600
        parts.append(remaining.assign(broadcast_date=date, hours=hours))
        remaining = remaining.loc[remaining.end_ts.gt(end)].copy()
        remaining['start_ts'] = end.loc[remaining.index]
    return pd.concat(parts, ignore_index=True)

raw_days = by_broadcast_day(acr[['tv_id','start_ts','end_ts']])
acr_days = by_broadcast_day(edited[['tv_id','start_ts','end_ts','session_type','network_code','input_source']])
raw_in_tab = raw_days.groupby('broadcast_date').tv_id.nunique()
in_tab = acr_days.groupby('broadcast_date').tv_id.nunique().reindex(days, fill_value=0)
print('TVs:', acr.tv_id.nunique())
print('Sessions:', len(acr))
print('Broadcast days:', len(raw_in_tab))
print('Average TVs in-tab per day:', f'{in_tab.mean():.1f}')
print('Hours per TV per day, raw:', f'{raw_days.hours.sum() / raw_in_tab.sum():.2f}')
print('Hours per TV per day, after edit rule:', f'{acr_days.hours.sum() / in_tab.sum():.2f}')
print('Share of sessions dropped:', f'{1 - len(edited) / len(acr):.3f}')

TVs: 29940
Sessions: 6392981
Broadcast days: 14
Average TVs in-tab per day: 29090.1
Hours per TV per day, raw: 4.73
Hours per TV per day, after edit rule: 4.70
Share of sessions dropped: 0.299


## Task 1.3 — What the TV could not see

In [3]:
hours_by_type = acr_days.groupby('session_type').hours.sum() / acr_days.hours.sum()
hours_by_input = acr_days.groupby('input_source').hours.sum() / acr_days.hours.sum()
display(hours_by_type.rename('Share of hours').round(3).to_frame())
display(hours_by_input.rename('Share of hours').round(3).to_frame())
print('Share of TV-on hours unknown (%):', f'{100 * hours_by_type["unknown"]:.1f}')

,Share of hours
session_type,
app,0.257
dvr,0.025
live,0.322
unknown,0.396


,Share of hours
input_source,
antenna,0.086
built_in_app,0.307
cable_box,0.276
game_or_disc,0.024
satellite_box,0.076
streaming_device,0.231


Share of TV-on hours unknown (%): 39.6


**Answer:** A real ACR feed may have a smaller fingerprint library relative to the variety of content on screens. Viewing from an unrecognized external stream and time spent playing a console game or watching a disc can both be logged as TV on without an identified network.

## Task 2.1 — Match

In [4]:
tv_keys = acr[['tv_id','hashed_ip']].drop_duplicates()
assert tv_keys.tv_id.is_unique
ip_counts = hh_attributes.groupby('hashed_ip').household_id.nunique()
tv_keys['households'] = tv_keys.hashed_ip.map(ip_counts).fillna(0)
unique_homes = hh_attributes.loc[hh_attributes.hashed_ip.map(ip_counts).eq(1)].drop_duplicates('hashed_ip')
matched = tv_keys.loc[tv_keys.households.eq(1)].merge(unique_homes, on='hashed_ip', validate='many_to_one')
print('Matched to one household (%):', f'{100 * tv_keys.households.eq(1).mean():.1f}')
print('Shared IP, unresolved (%):', f'{100 * tv_keys.households.gt(1).mean():.1f}')
print('Not matched (%):', f'{100 * tv_keys.households.eq(0).mean():.1f}')

Matched to one household (%): 65.2
Shared IP, unresolved (%): 11.8
Not matched (%): 22.9


**Answer:** Multiple households can share an IP address, for example through a building network. A household’s IP can also change, so a hash preserves the ambiguity of the original identifier.

## Task 2.2 — Profile

In [5]:
weights = daily_weights.copy()
weights['broadcast_date'] = pd.to_datetime(weights.broadcast_date).dt.normalize()
weights = weights.loc[weights.broadcast_date.isin(days)]
hh_weights = weights.loc[weights.resp_type.eq('H')]
monthly_hh = hh_weights.groupby('respondent_id').weight.mean()
profile_codes = viewer_attributes[['respondent_id','attr_id']].merge(attributes[['attr_id','attr_group','attr_name']], on='attr_id', validate='many_to_one')
region_lookup = profile_codes.loc[profile_codes.attr_group.eq('Census region')].set_index('respondent_id').attr_name
income_lookup = profile_codes.loc[profile_codes.attr_group.eq('Household income')].set_index('respondent_id').attr_name
income_names = attributes.loc[attributes.attr_group.eq('Household income')].sort_values('attr_id').attr_name
income_groups = dict(zip(income_names, ['Under 50k']*2 + ['50–100k']*2 + ['100k+']*2))
panel_hh = monthly_hh.to_frame().assign(region=region_lookup, income=income_lookup.map(income_groups))
matched['income'] = matched.income_band.map(income_groups)
assert panel_hh[['region','income']].notna().all().all() and matched.income.notna().all()
profiles = {}
for name in ['region','income']:
    footprint = matched[name].value_counts(normalize=True)
    panel_share = panel_hh.groupby(name).weight.sum() / panel_hh.weight.sum()
    profile = pd.DataFrame({'Footprint share':footprint, 'Panel share':panel_share})
    profile['Index'] = 100 * profile['Footprint share'] / profile['Panel share']
    profiles[name] = profile
    display(profile.style.format({'Footprint share':'{:.3f}', 'Panel share':'{:.3f}', 'Index':'{:.1f}'}))
display(matched.head_age_band.value_counts(normalize=True).round(3).to_frame('Matched footprint share'))

,Footprint share,Panel share,Index
region,,,
Midwest,0.201,0.210,96.0
Northeast,0.150,0.170,88.0
South,0.383,0.379,101.0
West,0.266,0.241,110.4


,Footprint share,Panel share,Index
income,,,
100k+,0.421,0.350,120.1
50–100k,0.320,0.289,110.8
Under 50k,0.259,0.361,71.8


,Matched footprint share
head_age_band,
35-49,0.348
50-64,0.263
18-34,0.196
65+,0.192


**Answer:** The West was overrepresented (index 110.4), the Northeast underrepresented (88.0), and the South close to parity (101.0). Households earning 100k or more were overrepresented (120.1), while households below 50k were underrepresented (71.8). Smart-TV ownership and the choice to opt in can both vary by income and location.

## Task 2.3 — Ratings

In [ ]:
comparison = reference_ratings.merge(programs, on='telecast_id', validate='one_to_one')
comparison['broadcast_date'] = pd.to_datetime(comparison.broadcast_date).dt.normalize()
for col in ['start_ts','end_ts']:
    comparison[col] = pd.to_datetime(comparison[col])

def watched_ids(sessions, episode, id_column):
    overlap = sessions.loc[sessions.network_code.eq(episode.network_code)
                           & sessions.start_ts.lt(episode.end_ts)
                           & sessions.end_ts.gt(episode.start_ts)]
    minutes = (overlap.end_ts.clip(upper=episode.end_ts)
               - overlap.start_ts.clip(lower=episode.start_ts)).dt.total_seconds() / 60
    totals = minutes.groupby(overlap[id_column]).sum()
    return totals.index[totals.ge(1)]

live = edited.loc[edited.session_type.eq('live')]
comparison['TVs HH rating'] = [100 * len(watched_ids(live, row, 'tv_id')) / in_tab.loc[row.broadcast_date]
                              for row in comparison.itertuples(index=False)]
comparison['Ratio'] = comparison['TVs HH rating'] / comparison.hh_rating
comparison['Gap'] = comparison['TVs HH rating'] - comparison.hh_rating
rating_cols = ['program_name','network_code','TVs HH rating','hh_rating','Ratio','Gap']
display(comparison[rating_cols].style.format({'TVs HH rating':'{:.1f}','hh_rating':'{:.1f}','Ratio':'{:.3f}','Gap':'{:+.1f}'}))

**Answer:** Every TV rating fell below the reference. NBC football and FOX baseball had the largest gaps in rating points (11.9 and 9.3). Bravo was closest by points, while Hannity was furthest off proportionally (ratio 0.359) and NBC football and Grotesquerie were closest proportionally (about 0.641). The larger relative shortfalls for Hannity and NCIS are consistent with weaker coverage of older-skewing audiences given the task’s audience descriptions, although the region and income profile alone cannot establish the cause.

## Task 3.1 — Hours per home

In [ ]:
ten = programs.network_code.unique()
acr_live = acr_days.loc[acr_days.session_type.eq('live') & acr_days.network_code.isin(ten)]
panel_sessions = viewings.loc[viewings.network_code.isin(ten) & viewings.resp_type.isin(['H','R','L']), ['respondent_id','resp_type','network_code','start_ts','end_ts']].copy()
for col in ['start_ts','end_ts']:
    panel_sessions[col] = pd.to_datetime(panel_sessions[col])
panel_days = by_broadcast_day(panel_sessions.loc[panel_sessions.resp_type.eq('H')])
panel_days = panel_days.merge(hh_weights[['respondent_id','broadcast_date','weight']], on=['respondent_id','broadcast_date'], validate='many_to_one')
panel_days['weighted_hours'] = panel_days.hours * panel_days.weight
tv_hours = acr_live.hours.sum() / in_tab.sum()
panel_hours = panel_days.weighted_hours.sum() / hh_weights.weight.sum()
print('Live hours per TV per day (ten networks):', f'{tv_hours:.2f}')
print('Live hours per panel household per day (ten networks):', f'{panel_hours:.2f}')
print('Panel minus TV hours:', f'{panel_hours - tv_hours:.2f}')

**Answer:** The TVs recorded 0.87 live hours per day across the ten networks, compared with 1.92 hours per weighted panel household, a difference of 1.05 hours. One opted-in set misses viewing on other sets that the panel can capture.

## Task 3.2 — Network shares

In [ ]:
tv_net = acr_live.groupby('network_code').hours.sum()
panel_net = panel_days.groupby('network_code').weighted_hours.sum()
network_shares = pd.DataFrame({'TV share': tv_net / tv_net.sum(), 'Panel share': panel_net / panel_net.sum()}).fillna(0)
network_shares['Difference'] = network_shares['TV share'] - network_shares['Panel share']
display(network_shares.sort_values('Difference', ascending=False).round(3))

# ABC and FOX have larger shares in the footprint; FNC and CBS have smaller shares.

## Task 3.3 — What each can see

**Answer:** The panel identifies individual viewers, provides person demographics and covers multiple measured sets in each household. The ACR feed records screen use on more devices, includes identified networks beyond the ten in the panel files and logs app or unknown screen time even when it cannot identify the content.

## Task 4.1 — Daily plots

In [ ]:
big = ['CBS','NBC','ABC','FOX','FNC']
big_daily = (acr_live.loc[acr_live.network_code.isin(big)]
             .groupby(['broadcast_date','network_code']).hours.sum()
             .unstack(fill_value=0).reindex(index=days, columns=big, fill_value=0))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
in_tab.plot(ax=axes[0], marker='o', title='TVs in-tab by broadcast day', ylabel='TVs')
big_daily.plot(ax=axes[1], marker='o', title='Identified live hours on the five big networks', ylabel='Hours', xlabel='Broadcast date')
fig.tight_layout()
plt.show()

## Task 4.2 — Spike and broken day

In [ ]:
spike_date, spike_network = big_daily.stack().idxmax()
program_dates = pd.to_datetime(programs.broadcast_date).dt.normalize()
spike_programs = programs.loc[program_dates.eq(spike_date) & programs.network_code.eq(spike_network)].copy()
for col in ['start_ts','end_ts']:
    spike_programs[col] = pd.to_datetime(spike_programs[col])
spike_sessions = acr_live.loc[acr_live.broadcast_date.eq(spike_date) & acr_live.network_code.eq(spike_network)]
spike_hours = []
for row in spike_programs.itertuples(index=False):
    active = spike_sessions.loc[spike_sessions.start_ts.lt(row.end_ts) & spike_sessions.end_ts.gt(row.start_ts)]
    overlap = active.end_ts.clip(upper=row.end_ts) - active.start_ts.clip(lower=row.start_ts)
    spike_hours.append(overlap.dt.total_seconds().sum() / 3600)
spike_programs['Identified live hours'] = spike_hours
quality = pd.DataFrame({'TVs in-tab':in_tab, 'Hours':acr_days.groupby('broadcast_date').hours.sum()})
quality['Hours per TV'] = quality.Hours / quality['TVs in-tab']
quality['TV count deviation'] = (quality['TVs in-tab'] / quality['TVs in-tab'].median() - 1).abs()
broken_day = quality['TV count deviation'].idxmax()
print('Biggest spike (date, network):', f'{spike_date.date()}, {spike_network}')
print('Broken day (date):', broken_day.date())
display(spike_programs[['program_name','Identified live hours']].nlargest(5,'Identified live hours').round(2))
display(quality.round(2))

**Answer:** The biggest spike was ABC on October 21, driven by Monday Night Football (about 6,991 identified live hours). On October 22, the in-tab count fell from 29,940 to 18,041, while hours per in-tab TV remained near weekday levels (4.40). Missing devices suppress the daily total, and ratings can move in either direction if the remaining TVs watch differently. Flag the day and exclude it from trend comparisons until the vendor restores and validates coverage.

## Task 5.1 — Sampling error vs bias

In [ ]:
se_table = comparison.loc[comparison.program_name.str.contains('NFL Football|Sunday Night Football|Hannity|Unsellable Houses', case=False, regex=True)].copy()
hh_counts = hh_weights.groupby('broadcast_date').respondent_id.nunique()
p = se_table.hh_rating / 100
se_table['SE footprint'] = 100 * np.sqrt(p * (1-p) / se_table.broadcast_date.map(in_tab))
se_table['SE panel'] = 100 * np.sqrt(p * (1-p) / se_table.broadcast_date.map(hh_counts))
se_table['Panel SE / footprint SE'] = se_table['SE panel'] / se_table['SE footprint']
display(se_table[['program_name','SE footprint','SE panel','Gap','Panel SE / footprint SE']].round(3))

**Answer:** The footprint sampling SE was about 2.8 times smaller than the panel SE for all three telecasts. The observed gaps were much larger than either SE. For NBC football the gap was 11.89 points, compared with SEs of 0.27 and 0.75 points. More devices reduce random sampling error, but coverage and selection bias require calibration.

## Task 5.2 — Answer to the planner

**Answer:** Today I can give you a 21.2% rating for the October 27 NBC game among in-tab opted-in TVs. The household currency reference is 33.1%, a gap of 11.9 points. The feed measures a single set and cannot identify the people watching or capture other sets in the home. App content may be unidentified, opted-out homes are outside the footprint, and unmatched devices cannot be profiled by household attributes. Before using this feed for planning, I would validate coverage, match quality and ratings against the panel and currency, then calibrate the footprint and check the residual bias.

## Bonus

In [ ]:
game = comparison.loc[comparison.network_code.eq('NBC') & comparison.broadcast_date.eq(pd.Timestamp('2024-10-27'))].iloc[0]
ages = viewer_attributes.loc[viewer_attributes.attr_id.eq(300)].set_index('respondent_id').value
target = ages.index[ages.between(25, 54)]
watched = watched_ids(panel_sessions, game, 'respondent_id')
game_weights = weights.loc[weights.broadcast_date.eq(game.broadcast_date)]
game_hh = game_weights.loc[game_weights.resp_type.eq('H')]
game_target = game_weights.loc[game_weights.resp_type.isin(['R','L']) & game_weights.respondent_id.isin(target)]
vpvh = game_target.loc[game_target.respondent_id.isin(watched), 'weight'].sum() / game_hh.loc[game_hh.respondent_id.isin(watched), 'weight'].sum()
boosted_rating = game['TVs HH rating'] * vpvh * game_hh.weight.sum() / game_target.weight.sum()
print('VPVH A25-54:', f'{vpvh:.3f}')
print('A25-54 rating (persons):', f'{boosted_rating:.1f}')

**Answer:** This estimate assumes that a viewing household in the TV footprint has the same number of A25–54 viewers per viewing household as in the weighted panel for this game. It also assumes device tuning is a useful proxy for household viewing. Coverage and selection bias can carry through to the person estimate.